In [74]:
import pandas as pd
import re

# Charger le df concaténé des deux législatures
df = pd.read_csv(
    "../data/interim/extract_15_16_concat.csv",
    low_memory=False,
    dtype={
        "id_orateur": str  # éviter identification en float avant d'avoir ajouté le "PA"
    },
)

print("Shape du df chargé : ", df.shape)

# Changer les missing values pour non_précisé (majoritaire) dans Code_parole
df["code_parole"] = df["code_parole"].fillna("non_précisé")

# NOTE : après test ne semble pas dramatique de ne pas prendre en compte
# le df["id_orateur"] = "PA" + df["id_orateur"] : seules 2 lignes changent
# les recodages "manuels" de PA repérés par ailleurs (voir autre notebook)
# ne changent rien non plus ici
# cf surtout des interruptions et ne change pas grand chose au regroup d'interventions
# + quand erreur pas forcément de changement d'ID entre ou d'interv.

# # Mais par principe si on veut garder :
# # Stabiliser le id_orateur pour être au format AN
# df["id_orateur"] = "PA" + df["id_orateur"]
# # Remplacer les valeurs manquantes de id_acteur par id_orateur quand disponible
# df["id_acteur_originel"] = df["id_acteur"]  # garder une trace
# df["id_acteur"] = df["id_acteur"].combine_first(df["id_orateur"])

Shape du df chargé :  (1127838, 29)


CG : Shape du df regroupé :  (968928, 34)


CG + ID : Shape du df regroupé :  (968930, 35)


CG + ID + NA : Shape du df regroupé :  (971491, 35)

In [ ]:
# TODO voir si devient pas trop conservateur avec en plus le code parole ????
# En gros : # avant d'ajouter condition code parole Shape du df regroupé :  (971491, 35)
# et après : (981871, 33)

"""
==========================
Regroupe les lignes de l'extraction CSV pour fusionner les interventions
d'un même orateur interrompues par des INTERRUPTION_1_10.

Sortie : un CSV entrelacé avec :
  - une ligne par groupe d'intervention fusionnée (texte concaténé)
  - les informations sur le nombre de fragments, d'interruptions reçues, etc.
  - les lignes INTERRUPTION conservées telles quelles, intercalées dans l'ordre
"""

# ---------------------------------------------------------------------------
# Paramètres
# ---------------------------------------------------------------------------

# Codes considérés comme interruptions (conservés tels quels dans la sortie)
CODES_INTERRUPTION = {"INTERRUPTION_1_10"}

# Colonnes invariantes dans un groupe (on garde la valeur de la 1ère ligne)
COLS_META = [
    "uid",
    "SeanceRef",
    "SessionRef",
    "dateSeance",
    "dateSeanceJour",
    "numSeanceJour",
    "numSeance",
    "typeAssemblee",
    "legislature",
    "session",
    "nomFichierJo",
    "presidentSeance",
    "point_titre",
    "point_type",
    "valeur_ptsodj",
    "ordinal_prise",
    "ordre_absolu_seance",
    "id_acteur",
    "id_mandat",
    "code_grammaire",
    "code_style",
    "code_parole",
    "id_syceron",
    "roledebat",
    "nom_orateur",
    "qualite_orateur",
    "id_orateur",
    "stime",
]

# ---------------------------------------------------------------------------
# Fonction principale
# ---------------------------------------------------------------------------


def regrouper(df: pd.DataFrame) -> pd.DataFrame:
    """
    Prend un DataFrame trié par (uid, ordre_absolu_seance) et retourne
    un DataFrame entrelacé :
      - lignes d'intervention fusionnées (nb_fragments >= 1)
      - lignes d'interruption conservées telles quelles (nb_fragments = NaN)
    """
    cols_utiles = list(dict.fromkeys(COLS_META + ["texte"]))
    work = df[cols_utiles].copy()

    work["uid_norm"] = work["uid"].fillna("").astype(str)
    work["id_acteur_norm"] = work["id_acteur"].fillna("").astype(str)
    work["code_grammaire_norm"] = work["code_grammaire"].fillna("").astype(str)
    work["code_parole_norm"] = work["code_parole"].fillna("").astype(str)  # TODO : test
    work["texte_norm"] = work["texte"].fillna("").astype(str)

    work = work.sort_values(["uid_norm", "ordre_absolu_seance"]).reset_index(drop=True)

    resultats = []  # liste finale (interventions + interruptions)
    groupe = None  # groupe en cours d'accumulation
    buffer_interruptions = []  # interruptions entre deux fragments du même orateur

    def ligne_sortie_depuis_base(base_row: dict) -> dict:
        r = {col: base_row[col] for col in cols_utiles}
        r["nb_fragments"] = pd.NA
        r["nb_interruptions_recues"] = pd.NA
        r["a_ete_interrompu"] = pd.NA
        # r["codes_gram_fragments"] = pd.NA # ie pour traçabilité si enlève condition
        # r["codes_parole_fragments"] = pd.NA # ie pour traçabilité si enlève condition
        r["id_syceron_fragments"] = pd.NA
        # r["changement_code_grammaire"] = pd.NA # ie pour traçabilité si enlève condition
        # r["changement_code_parole"] = pd.NA # ie pour traçabilité si enlève condition
        return r

    def clore_groupe(g: dict) -> dict:
        """
        Finalise un groupe. Les interruptions du buffer seront émises APRÈS dans le flux.
        """
        row = g["premiere_ligne"].copy()
        row["texte"] = " ".join(
            g["textes"]
        )  # on prend les textes norm pour éviter les NaN
        row["nb_fragments"] = g["nb_fragments"]
        row["nb_interruptions_recues"] = g["nb_interruptions_recues"]
        row["a_ete_interrompu"] = g["nb_interruptions_recues"] > 0
        # row["codes_gram_fragments"] = "|".join(g["codes_grammaire"]) # ie pour traçabilité si enlève condition
        # row["codes_parole_fragments"] = "|".join(g["codes_parole"]) # ie pour traçabilité si enlève condition
        row["id_syceron_fragments"] = "|".join(g["codes_syceron"])
        # row["changement_code_grammaire"] = len(set(g["codes_grammaire"])) > 1 # ie pour traçabilité si enlève condition
        # row["changement_code_parole"] = len(set(g["codes_parole"])) > 1 # ie pour traçabilité si enlève condition
        return row

    records = work.to_dict("records")

    for row in records:
        cg = row["code_grammaire_norm"]
        cp = row["code_parole_norm"]  # TODO : test
        acteur_str = row["id_acteur_norm"]
        uid_str = row["uid_norm"]
        syc = str(row["id_syceron"]) if pd.notna(row["id_syceron"]) else ""

        # --- Cas 1 : interruption ---
        if cg in CODES_INTERRUPTION:
            if groupe is not None:
                # L'interruption est dans le contexte d'un groupe ouvert :
                # on l'ajoute au buffer (elle sera émise si le même orateur reprend)
                buffer_interruptions.append(row)
                groupe["nb_interruptions_recues"] += 1
            else:
                # Interruption hors contexte (cas rare) : on l'émet directement
                resultats.append(ligne_sortie_depuis_base(row))
            continue

        # --- Cas 2 : intervention principale ---
        if (
            groupe is not None
            and buffer_interruptions  # on regroupe que si bien interrompu (et pas parle 2 fois de suite)
            and acteur_str != ""  # cf les nan convertis en ""
            and groupe["id_acteur"] == acteur_str
            and groupe["uid"] == uid_str
            and groupe["codes_grammaire"][-1] == cg
            and groupe["codes_parole"][-1] == cp
        ):
            # Même orateur, même séance, mêmes codes, avec interruption -> on fusionne
            groupe["textes"].append(row["texte_norm"])
            groupe["codes_grammaire"].append(cg)
            groupe["codes_parole"].append(cp)  # TODO : test
            groupe["codes_syceron"].append(syc)
            groupe["nb_fragments"] += 1
        else:
            # Nouvel orateur ou nouvelle séance ou changement de code_grammaire
            if groupe is not None:
                # Clore le groupe précédent
                resultats.append(clore_groupe(groupe))
                # Et les interruptions en buffer suivent le groupe
                for irr in buffer_interruptions:
                    resultats.append(ligne_sortie_depuis_base(irr))
                buffer_interruptions = []

            groupe = {
                "uid": uid_str,
                "id_acteur": acteur_str,
                "premiere_ligne": {col: row[col] for col in cols_utiles},
                "textes": [row["texte_norm"]],
                "codes_grammaire": [cg],
                "codes_parole": [cp],
                "codes_syceron": [syc],
                "nb_fragments": 1,
                "nb_interruptions_recues": 0,
            }
    # Clore le dernier groupe
    if groupe is not None:
        resultats.append(clore_groupe(groupe))
        for irr in buffer_interruptions:
            resultats.append(ligne_sortie_depuis_base(irr))

    return pd.DataFrame(resultats)


In [81]:
df_group_bis = regrouper(df)
print("Shape du df regroupé : ", df_group_bis.shape)

Shape du df regroupé :  (981871, 33)


In [82]:
df_group_bis.to_csv("../data/interim/TEST_INTERRUPTIONS_BIS.csv", index=False)

In [83]:
df_group_bis.columns

Index(['uid', 'SeanceRef', 'SessionRef', 'dateSeance', 'dateSeanceJour',
       'numSeanceJour', 'numSeance', 'typeAssemblee', 'legislature', 'session',
       'nomFichierJo', 'presidentSeance', 'point_titre', 'point_type',
       'valeur_ptsodj', 'ordinal_prise', 'ordre_absolu_seance', 'id_acteur',
       'id_mandat', 'code_grammaire', 'code_style', 'code_parole',
       'id_syceron', 'roledebat', 'nom_orateur', 'qualite_orateur',
       'id_orateur', 'stime', 'texte', 'nb_fragments',
       'nb_interruptions_recues', 'a_ete_interrompu', 'id_syceron_fragments'],
      dtype='object')

# TESTS

In [91]:
# TODO : ÇA DESSOUS C'EST DE LA MERDE, TESTE EN FAISANT LES DEUX EXPORTS ET VA VOIR LES DIFF FEIGNASSE


In [90]:
# Diagnostic avancé : séquences de fragments séparées par code_parole

def diagnostic_sequences_code_parole(df: pd.DataFrame) -> pd.DataFrame:
    """
    Identifie toutes les séquences (>=2 fragments) qui auraient été fusionnées sans la condition
    code_parole, mais qui sont désormais séparées à cause d'un changement de code_parole entre fragments,
    avec des interruptions entre.
    """
    work = df.sort_values(["uid", "ordre_absolu_seance"]).reset_index(drop=True)
    is_interrupt = work["code_grammaire"].isin(CODES_INTERRUPTION)
    main = work[~is_interrupt].copy().reset_index(drop=True)

    # On crée un identifiant de "groupe potentiel" sans tenir compte de code_parole
    rupture = (
        (main["id_acteur"] != main["id_acteur"].shift(1)) |
        (main["uid"] != main["uid"].shift(1)) |
        (main["code_grammaire"] != main["code_grammaire"].shift(1)) |
        (main["id_acteur"].isna()) |
        (main["id_acteur"].fillna("") == "")
    )
    main["groupe_potentiel"] = rupture.cumsum()

    # Pour chaque groupe potentiel, on regarde s'il y a plusieurs valeurs de code_parole
    groupes = main.groupby("groupe_potentiel")
    res = []
    for gid, group in groupes:
        if len(group) < 2:
            continue  # pas une séquence
        if group["code_parole"].nunique() == 1:
            continue  # pas de rupture sur code_parole
        # On cherche les interruptions entre chaque fragment
        frag_idx = group.index.values
        interruptions = []
        for i in range(len(frag_idx)-1):
            frag1 = main.loc[frag_idx[i]]
            frag2 = main.loc[frag_idx[i+1]]
            # Y a-t-il une interruption entre frag1 et frag2 ?
            irr = work[
                (work["uid"] == frag1["uid"])
                & (work["ordre_absolu_seance"] > frag1["ordre_absolu_seance"])
                & (work["ordre_absolu_seance"] < frag2["ordre_absolu_seance"])
                & is_interrupt
            ]
            interruptions.append(len(irr) > 0)
        if all(interruptions):
            # On ne garde que les séquences où chaque rupture est bien séparée par une interruption
            res.append(group)
    if not res:
        print("Aucune séquence complexe trouvée.")
        return pd.DataFrame()
    result = pd.concat(res)
    print(f"{result['groupe_potentiel'].nunique()} séquences complexes concernées.")
    return result

# Exécution et export du diagnostic avancé
df_diag_seq = diagnostic_sequences_code_parole(df)
df_diag_seq.to_csv("../data/interim/diagnostic_sequences_code_parole.csv", index=False)
df_diag_seq.head()

1137 séquences complexes concernées.


,uid,SeanceRef,SessionRef,dateSeance,dateSeanceJour,numSeanceJour,numSeance,typeAssemblee,legislature,session,...,code_style,code_parole,id_syceron,roledebat,nom_orateur,qualite_orateur,id_orateur,stime,texte,groupe_potentiel
897,CRSANR5L15S2017E1N006,NaN,NaN,20170710213000000,lundi 10 juillet 2017,2,6,AN,15,Première session extraordinaire 2017,...,NORMAL,AVIS_GVT_1_20,984999,NaN,Mme Muriel Pénicaud,ministre du travail,717169,NaN,Même avis que le rapporteur de la commission. ...,527
898,CRSANR5L15S2017E1N006,NaN,NaN,20170710213000000,lundi 10 juillet 2017,2,6,AN,15,Première session extraordinaire 2017,...,NORMAL,AVIS_GVT_1_20,984918,NaN,Mme Muriel Pénicaud,ministre,717169,NaN,"Je reviens un instant, dans le même esprit qu’...",527
899,CRSANR5L15S2017E1N006,NaN,NaN,20170710213000000,lundi 10 juillet 2017,2,6,AN,15,Première session extraordinaire 2017,...,NORMAL,AVIS_GVT_1_20,985002,NaN,Mme Muriel Pénicaud,ministre,717169,NaN,Certaines branches déterminent le niveau des f...,527
900,CRSANR5L15S2017E1N006,NaN,NaN,20170710213000000,lundi 10 juillet 2017,2,6,AN,15,Première session extraordinaire 2017,...,NORMAL,PAROLE_1_2,985011,NaN,Mme Muriel Pénicaud,ministre,717169,NaN,Je vous ai écouté : je vous remercie de m’écou...,527
901,CRSANR5L15S2017E1N006,NaN,NaN,20170710213000000,lundi 10 juillet 2017,2,6,AN,15,Première session extraordinaire 2017,...,NORMAL,PAROLE_1_2,985020,NaN,Mme Muriel Pénicaud,ministre,717169,NaN,"Nous avons besoin aujourd’hui, de sécuriser l’...",527


GOGO GADGETO IA :

Le nombre de lignes du DataFrame augmente fortement quand on ajoute la condition sur code_parole (on passe de 971 491 à 981 871, soit +10 380 lignes).
Mais seules 1 188 paires sont des cas où deux fragments auraient été fusionnés sans la condition sur code_parole (et ne le sont plus à cause de cette condition).
Pourquoi ?
Parce que chaque “rupture” sur code_parole crée une nouvelle ligne, mais beaucoup de ces ruptures ne sont pas des paires strictes : il peut y avoir des interruptions, des séquences plus longues, ou des cas où la rupture ne concerne pas juste deux fragments consécutifs.
La fonction diagnostique ne remonte que les paires strictes (fragment 1, interruption, fragment 2, même acteur, même séance, même code_grammaire, mais code_parole différent).

Donc :

Le surplus de lignes vient de toutes les ruptures, pas seulement des paires “simples” détectées par le diagnostic.
Le diagnostic donne un minimum garanti de cas “simples” affectés, mais il peut y avoir des cas plus complexes (plus de deux fragments consécutifs, etc.).
C’est donc cohérent.
Si tu veux aussi les cas plus complexes (séquences de 3 fragments ou plus), il faudrait un diagnostic plus avancé.

In [87]:
# # TODO fonction TER
# """
# ==========================
# Regroupe les lignes d'un CSV parlementaire pour fusionner les interventions
# d'un même orateur interrompues par des INTERRUPTION_*.

# Sortie : un CSV entrelacé avec :
#   - une ligne par groupe d'intervention fusionnée (texte concaténé)
#   - les lignes INTERRUPTION conservées telles quelles, placées APRÈS leur groupe
# """

# PATTERN_INTERRUPTION = "INTERRUPTION"


# # ---------------------------------------------------------------------------
# # Fonction principale (vectorisée)
# # ---------------------------------------------------------------------------


# def regrouper(df: pd.DataFrame) -> pd.DataFrame:
#     """
#     Regroupe les interventions interrompues de manière vectorisée.

#     Stratégie :
#       1. Séparer interventions principales et interruptions
#       2. Sur les interventions, détecter les ruptures de groupe
#          (changement d'acteur ou de séance) → groupe_id cumulatif
#       3. Agréger par groupe_id (texte concaténé, first pour le reste)
#       4. Propager le groupe_id aux interruptions via ffill
#       5. Reconstruire le flux : intervention puis ses interruptions,
#          triés par (uid, ordre_absolu_seance du 1er fragment, type)
#     """
#     df = df.sort_values(["uid", "ordre_absolu_seance"]).reset_index(drop=True)

#     is_interrupt = df["code_grammaire"].str.contains(PATTERN_INTERRUPTION, na=False)
#     main = df[~is_interrupt].copy()
#     interrupts = df[is_interrupt].copy()

#     # ------------------------------------------------------------------
#     # 1. Calculer les groupe_id sur les interventions principales
#     # ------------------------------------------------------------------
#     acteur_norm = main["id_acteur"].fillna("").astype(str)
#     uid_norm = main["uid"].fillna("").astype(str)
#     code_norm = main["code_grammaire"].fillna("").astype(str)

#     # Rupture si : changement d'acteur, de séance, de code_grammaire,
#     # ou acteur vide (lignes sans id_acteur ne sont jamais fusionnées)
#     rupture = (
#         (acteur_norm != acteur_norm.shift(1))
#         | (uid_norm != uid_norm.shift(1))
#         | (code_norm != code_norm.shift(1))
#         | (acteur_norm == "")
#     )
#     main["groupe_id"] = rupture.cumsum()

#     # ------------------------------------------------------------------
#     # 2. Agréger les interventions par groupe
#     # ------------------------------------------------------------------
#     agg = {
#         c: "first"
#         for c in main.columns
#         if c not in ["texte", "code_grammaire", "groupe_id"]
#     }
#     agg["texte"] = lambda s: " ".join(s.dropna().astype(str))
#     agg["code_grammaire"] = lambda s: "|".join(s.dropna().astype(str))

#     grouped = main.groupby("groupe_id", sort=False).agg(agg).reset_index(drop=True)

#     # nb_fragments
#     grouped["nb_fragments"] = main.groupby("groupe_id", sort=False).size().values

#     # codes_fragments / changement_code_grammaire / code_grammaire (1er fragment)
#     grouped.rename(columns={"code_grammaire": "codes_fragments"}, inplace=True)
#     grouped["code_grammaire"] = grouped["codes_fragments"].str.split("|").str[0]
#     grouped["changement_code_grammaire"] = grouped["codes_fragments"].apply(
#         lambda s: len(set(s.split("|"))) > 1
#     )

#     # ------------------------------------------------------------------
#     # 3. Propager groupe_id aux interruptions via ffill
#     # ------------------------------------------------------------------
#     df["groupe_id"] = pd.NA
#     df.loc[~is_interrupt, "groupe_id"] = main["groupe_id"].values
#     df["groupe_id"] = df["groupe_id"].ffill()

#     # Compter les interruptions par groupe
#     interrupt_counts = (
#         df[is_interrupt]
#         .groupby("groupe_id", sort=False)
#         .size()
#         .rename("nb_interruptions_recues")
#         .reset_index()
#     )

#     # Rattacher le groupe_id au grouped pour le merge
#     groupe_ids = main.groupby("groupe_id", sort=False)["groupe_id"].first().values
#     grouped["groupe_id"] = groupe_ids
#     grouped = grouped.merge(interrupt_counts, on="groupe_id", how="left")
#     grouped["nb_interruptions_recues"] = (
#         grouped["nb_interruptions_recues"].fillna(0).astype(int)
#     )
#     grouped["a_ete_interrompu"] = grouped["nb_interruptions_recues"] > 0
#     grouped.drop(columns="groupe_id", inplace=True)

#     # ------------------------------------------------------------------
#     # 4. Reconstruire le flux ordonné
#     # ------------------------------------------------------------------
#     # tie-breaker : 0 = intervention, 1 = interruption
#     # → intervention toujours avant ses interruptions à même ordre
#     for col in [
#         "nb_fragments",
#         "nb_interruptions_recues",
#         "a_ete_interrompu",
#         "codes_fragments",
#         "changement_code_grammaire",
#     ]:
#         interrupts[col] = pd.NA

#     grouped["_sort_ordre"] = grouped["ordre_absolu_seance"]
#     grouped["_sort_type"] = 0

#     interrupts["_sort_ordre"] = interrupts["ordre_absolu_seance"]
#     interrupts["_sort_type"] = 1

#     result = (
#         pd.concat([grouped, interrupts], ignore_index=True)
#         .sort_values(["uid", "_sort_ordre", "_sort_type"])
#         .drop(columns=["_sort_ordre", "_sort_type"])
#         .reset_index(drop=True)
#     )

#     return result


# # ---------------------------------------------------------------------------
# # Diagnostic : changements de code_grammaire pour un même acteur
# # ---------------------------------------------------------------------------


# def diagnostic_changements_code(df: pd.DataFrame) -> pd.DataFrame:
#     """
#     À lancer sur le df ORIGINAL (avant regroupement).
#     Retourne les cas où un même acteur enchaîne deux code_grammaire différents
#     sans interruption entre eux — utile pour investiguer les <interExtraction>.
#     """
#     main = df[~df["code_grammaire"].str.contains(PATTERN_INTERRUPTION, na=False)].copy()
#     main = main.sort_values(["uid", "ordre_absolu_seance"])

#     main["prev_acteur"] = main["id_acteur"].shift(1)
#     main["prev_code"] = main["code_grammaire"].shift(1)
#     main["prev_uid"] = main["uid"].shift(1)

#     cas = main[
#         (main["id_acteur"] == main["prev_acteur"])
#         & (main["uid"] == main["prev_uid"])
#         & (main["code_grammaire"] != main["prev_code"])
#         & main["id_acteur"].notna()
#         & (main["id_acteur"] != "")
#     ][
#         [
#             "uid",
#             "ordre_absolu_seance",
#             "id_acteur",
#             "nom_orateur",
#             "prev_code",
#             "code_grammaire",
#             "texte",
#         ]
#     ].copy()

#     cas.columns = [
#         "uid",
#         "ordre",
#         "id_acteur",
#         "nom_orateur",
#         "code_precedent",
#         "code_courant",
#         "texte",
#     ]
#     return cas


In [ ]:
# df_group_ter = regrouper(df)
# print("Shape du df regroupé : ", df_group_ter.shape)

/var/folders/rq/xsj46x_s2rg87wdksm1_jl3c0000gn/T/ipykernel_4992/2392771363.py:79: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['groupe_id'] = df['groupe_id'].ffill()


Shape du df regroupé :  (971491, 35)


In [ ]:
# df_group_ter.to_csv("../data/interim/TEST_INTERRUPTIONS_TER.csv", index=False)

In [ ]:
# def diagnostic_changements_code_grammaire(df: pd.DataFrame) -> pd.DataFrame:
#     main = df[~df["code_grammaire"].isin(CODES_INTERRUPTION)].copy()
#     main = main.sort_values(["uid", "ordre_absolu_seance"])

#     main["prev_acteur"] = main["id_acteur"].shift(1)
#     main["prev_uid"] = main["uid"].shift(1)
#     main["prev_code_grammaire"] = main["code_grammaire"].shift(1)

#     cas = main[
#         (main["id_acteur"] == main["prev_acteur"])
#         & (main["uid"] == main["prev_uid"])
#         & (main["code_grammaire"] != main["prev_code_grammaire"])
#         & main["id_acteur"].notna()
#         & (main["id_acteur"] != "")
#     ][
#         [
#             "uid",
#             "ordre_absolu_seance",
#             "id_acteur",
#             "nom_orateur",
#             "prev_code_grammaire",
#             "code_grammaire",
#             "texte",
#         ]
#     ].copy()

#     return cas


# def diagnostic_changements_code_parole(df: pd.DataFrame) -> pd.DataFrame:
#     main = df[~df["code_grammaire"].isin(CODES_INTERRUPTION)].copy()
#     main = main.sort_values(["uid", "ordre_absolu_seance"])

#     main["prev_acteur"] = main["id_acteur"].shift(1)
#     main["prev_uid"] = main["uid"].shift(1)
#     main["prev_code_parole"] = main["code_parole"].shift(1)

#     cas = main[
#         (main["id_acteur"] == main["prev_acteur"])
#         & (main["uid"] == main["prev_uid"])
#         & (main["code_parole"] != main["prev_code_parole"])
#         & main["id_acteur"].notna()
#         & (main["id_acteur"] != "")
#     ][
#         [
#             "uid",
#             "ordre_absolu_seance",
#             "id_acteur",
#             "nom_orateur",
#             "prev_code_parole",
#             "code_parole",
#             "code_grammaire",
#             "texte",
#         ]
#     ].copy()

#     return cas
